In [0]:
-- =====================================================
-- Summary TABLES FOR DASHBOARDS
-- Refresh after fact_reconciliation load
-- =====================================================

-- =====================================================
-- 1. DAILY SUMMARY TABLE
-- =====================================================
CREATE OR REPLACE TABLE payment_app.gold.daily_summary_table AS
SELECT
    date_key,
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN recon_flag = 'Y' THEN 1 ELSE 0 END) AS success_transactions,
    SUM(CASE WHEN recon_flag = 'N' THEN 1 ELSE 0 END) AS exception_transactions,
    ROUND(
        100.0 * SUM(CASE WHEN recon_flag = 'Y' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS success_rate_pct,
    SUM(txn_amount) AS total_txn_amount,
    SUM(credited_amount) AS total_credited_amount
FROM payment_app.gold.fact_reconciliation
GROUP BY date_key;


-- =====================================================
-- 2. SOURCE SUMMARY TABLE
-- =====================================================
CREATE OR REPLACE TABLE payment_app.gold.source_summary_table AS
SELECT
    date_key,
    txn_source,
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN recon_flag = 'Y' THEN 1 ELSE 0 END) AS success_transactions,
    SUM(CASE WHEN recon_flag = 'N' THEN 1 ELSE 0 END) AS exception_transactions,
    ROUND(
        100.0 * SUM(CASE WHEN recon_flag = 'Y' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS success_rate_pct,
    SUM(txn_amount) AS total_txn_amount,
    SUM(credited_amount) AS total_credited_amount
FROM payment_app.gold.fact_reconciliation
GROUP BY date_key, txn_source;


-- =====================================================
-- 3. BILLER SUMMARY TABLE
-- =====================================================
CREATE OR REPLACE TABLE payment_app.gold.biller_summary_table AS
SELECT
    f.date_key,
    b.biller_id,
    b.biller_name,
    b.biller_category,

    COUNT(*) AS total_transactions,

    SUM(CASE WHEN f.recon_flag = 'Y' THEN 1 ELSE 0 END) AS success_transactions,

    SUM(CASE WHEN f.recon_flag = 'N' THEN 1 ELSE 0 END) AS exception_transactions,

    ROUND(
        100.0 * SUM(CASE WHEN f.recon_flag = 'Y' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS success_rate_pct,

    SUM(f.txn_amount) AS total_txn_amount,
    SUM(f.credited_amount) AS total_credited_amount

FROM payment_app.gold.fact_reconciliation f
LEFT JOIN payment_app.gold.dim_biller b
    ON f.biller_key = b.biller_key

GROUP BY
    f.date_key,
    b.biller_id,
    b.biller_name,
    b.biller_category;


-- =====================================================
-- OPTIONAL VALIDATION QUERIES
-- =====================================================

SELECT * FROM payment_app.gold.daily_summary_table;
SELECT * FROM payment_app.gold.source_summary_table;
SELECT * FROM payment_app.gold.biller_summary_table;